# **Deep Natural Language Processing @ PoliTO**


---
**Teaching Assistant:** Ali Yassine

**Credits:** Moreno La Quatra

**Practice 5:** Machine Translation - Part 1

## **Machine Translation**

Machine Translation is a sub-field of Natural Language Processing that aims at translating a text from a source language to a target language. In this practice, we will experiment with a Transformer-based model for Machine Translation. Specifically, we will benchmark the performance of a pre-trained MT model on Italian-English and English-Italian translation tasks.

![](https://www.deepl.com/img/press/desktop_ENIT_2020-01.png)

In this practice we will use a data collection provided by [tatoeba](https://tatoeba.org/). The following cell download a subset of the data collection, containing parallel Italian-English sentences.


In [1]:
%%capture
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P6/train_it_en.tsv
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P6/test_it_en.tsv

### **Question 1: Parsing data**

The first step is to parse the data collection to generate a list of sentence pairs. The data are provided in `tsv` format, where each line contains a sentence pair in the following format:

`<source_language_sentence>\t<target_language_sentence>\n`

You are provided with a training and a test set. For this question you should parse both data splits and store them in your preferred data structure.

**Note:** store train and test set into separate data objects.

In [2]:
# your code here
import pandas as pd

train_df = pd.read_csv("train_it_en.tsv", sep='\t')
test_df = pd.read_csv("test-it-en.tsv", sep='\t')

it_sents_train = train_df['it_sent'].to_list()
it_sents_test = test_df['it_sent'].to_list()
en_sents_train = train_df['en_sent'].to_list()
en_sents_test = test_df['en_sent'].to_list()

### **Question 2: Pre-trained MT models**

Pre-trained MT models are released to the public to allow researchers to experiment with them. In this question you will load a pre-trained MT model and use it to translate sentences from Italian to English and vice-versa.

[EasyNMT](https://github.com/UKPLab/EasyNMT) is a Python library that provides an easy-to-use interface to pre-trained MT models. It provides a simple wrapper over HuggingFace transformers library for machine translation. In this question you will use EasyNMT to load a pre-trained MT model and translate sentences from Italian to English and vice-versa:

- Load the pre-trained model for a specific direction (e.g., Italian-English or English-Italian)
- Translate all the sentences in the test set from the source language to the target language.


**Note 1**: the choice for the MT model is up to you.

**Note 2**: store the translated sentences in both directions using the data structure of your choice.

In [3]:
%%capture
!pip install easynmt sacremoses

In [4]:
# your code here
import easynmt

model_id = "opus-mt"
model = easynmt.EasyNMT(model_id)

it_en_translations = model.translate(
    it_sents_test,
    source_lang="it",
    target_lang="en"
)

en_it_translations = model.translate(
    en_sents_test,
    source_lang="en",
    target_lang="it"
)

### **Question 3: BLEU and METEOR scores**

In this question you will evaluate the performance of your machine translation (MT) model using **two** evaluation metrics: **[BLEU evaluation metric](https://github.com/mjpost/sacrebleu)** and **[METEOR evaluation metric](https://huggingface.co/spaces/evaluate-metric/meteor)**. You **must** compute and report scores for both translation directions: `EN→IT` and `IT→EN`.

---

#### BLEU (Bilingual Evaluation Understudy)

**BLEU** measures how much the model’s translation overlaps with a reference translation by comparing shared **n-grams** (word sequences). It gives a precision-oriented score that rewards exact word matches.

- **Pros:** Fast, standardized, and good for large-scale comparisons.
- **Cons:** Only captures exact matches, ignoring synonyms or paraphrases; may not always align with human judgment.

> Use BLEU as implemented in `sacrebleu`.


---

#### METEOR (Metric for Evaluation of Translation with Explicit ORdering)

**METEOR** was developed to better reflect human judgment by allowing more flexible word matching. It aligns hypothesis and reference words using:

- **Exact matches**
- **Stem matches** (e.g., *run* ↔ *running*)
- **Synonyms and paraphrases**

It then combines these matches into a single score with penalties for disordered or fragmented output.

- **Pros:** More linguistically aware; correlates better with human evaluations.
- **Cons:** Slower to compute; depends on external lexical resources.

> Compute METEOR using `evaluate` or `nltk`.

---

The following cell installs the `sacrebleu`, `nltk`, and `evaluate` libraries that can be used to compute these metrics.

In [5]:
%%capture
!pip install sacrebleu evaluate nltk

In [6]:
# your code here
from sacrebleu import corpus_bleu

bleu_it_en = corpus_bleu(it_en_translations, [en_sents_test])
print(f"IT->EN BLEU Score: {bleu_it_en.score}")

bleu_en_it = corpus_bleu(en_it_translations, [it_sents_test])
print(f"EN->IT BLEU Score: {bleu_en_it.score}")

IT->EN BLEU Score: 71.6386453129808
EN->IT BLEU Score: 49.51485374240428


In [7]:
import evaluate

metric = evaluate.load("meteor")

result = metric.compute(
    predictions=it_en_translations,
    references=en_sents_test
)
print(f"IT->EN Meteor Score: {result['meteor']}")

result = metric.compute(
    predictions=en_it_translations,
    references=it_sents_test
)
print(f"EN->IT Meteor Score: {result['meteor']}")

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


IT->EN Meteor Score: 0.8694188547665864
EN->IT Meteor Score: 0.6824232847151652


### **Question 4: Comparison with another Pre-trained MT Model**

In this question, you will experiment with another pre-trained MT model and compare its performance with the model used in Question 2.

Follow the same translation procedure as before (EN→IT and IT→EN) and evaluate the results using the BLEU and METEOR metrics from Question 3.

Use [EasyNMT](https://github.com/UKPLab/EasyNMT) to load and run the pre-trained model.


In [11]:
# your code here
import easynmt
from sacrebleu import corpus_bleu
import evaluate

model_id_q4 = "m2m_100_1.2b"
model_q4 = easynmt.EasyNMT(model_id_q4)

it_en_translations_q4 = model_q4.translate(
    it_sents_test,
    source_lang="it",
    target_lang="en"
)

en_it_translations_q4 = model_q4.translate(
    en_sents_test,
    source_lang="en",
    target_lang="it"
)

bleu_it_en_q4 = corpus_bleu(
    it_en_translations_q4,
    [en_sents_test]
)
print(f"IT->EN BLEU Score: {bleu_it_en_q4.score}")

bleu_en_it_q4 = corpus_bleu(
    en_it_translations_q4,
    [it_sents_test]
)
print(f"EN->IT BLEU Score: {bleu_en_it_q4.score}")

metric_meteor = evaluate.load("meteor")

result_it_en_q4 = metric_meteor.compute(
    predictions=it_en_translations_q4,
    references=en_sents_test
)
print(f"IT->EN Meteor Score: {result_it_en_q4['meteor']}")

result_en_it_q4 = metric_meteor.compute(
    predictions=en_it_translations_q4,
    references=it_sents_test
)
print(f"EN->IT Meteor Score: {result_en_it_q4['meteor']}")

89.9kB [00:00, 707kB/s]                    


config.json:   0%|          | 0.00/909 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

IT->EN BLEU Score: 55.656860627169934
EN->IT BLEU Score: 44.26039264302649


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


IT->EN Meteor Score: 0.7645824436120915
EN->IT Meteor Score: 0.6559653808672443
